%md
# Example on how to let a list of missions available for download.
### Author: Dan Williams
### Email: dan@agdmalabs.com
### Date: 5/20/2026

*

In [ ]:
%pip install "typing_extensions>=4.14.0" git+https://github.com/AgDMALabs-Public/ag-vision-dataops.git

In [ ]:
import os
from dotenv import load_dotenv
from ag_vision.external_connections.hiphen import utils as hu

In [ ]:
dbutils.library.restartPython()

In [ ]:
# The final table will be filtered by this list.
contract_names = ['CIMMYT Wheat']

DB_VOLUME = "/Volumes/aps1_prod_gems_fg_catalog_7474649869966386/tier1_raw/data"
TABLE_NAME = "aps1_prod_gems_fg_catalog_7474649869966386.tier1_raw.hipen_missions"

In [ ]:
client_secret = dbutils.secrets.get(scope="hiphen",
                                    key="hiphen-api-key")

# for local dev
#load_dotenv()
#client_secret = os.getenv('HIPHEN_SECRET')
#assert client_secret is not None

SITE_ID = "cgiar-cloverfield-api"
CLIENT_ID = "cgiar-cloverfield-api"
DEFAULT_BASE_URL = "https://api.hiphen-cloverfield.com"

In [ ]:
print(client_secret)

In [ ]:
token_mgr = hu.TokenManager(base_url=DEFAULT_BASE_URL,
                            client_id=CLIENT_ID,
                            client_secret=client_secret)

token_mgr.get_headers()

In [ ]:
hi = hu.HiphenData(token_mgr=token_mgr)

In [ ]:
hi.list_contracts()

In [ ]:
hi.generate_contract_site_mission_table()

In [ ]:
hi.data_summary = hi.data_summary.fillna('UNKNOWN')

In [ ]:
hi.data_summary['contract'].unique()

In [ ]:
hi.add_plot_dir_to_summary_table(volume=DB_VOLUME)
hi.count_plot_images()

In [ ]:
hi.data_summary.sample(10)

In [ ]:
hi.data_summary = hi.data_summary[hi.data_summary['contract'].isin(contract_names)]

In [ ]:
spark_df = spark.createDataFrame(hi.data_summary)
spark_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(TABLE_NAME)